In [1]:
!pip install transformers torch accelerate scikit-learn datasets

In [2]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch

In [3]:
df = pd.read_csv("IMDB Dataset.csv", engine="python", on_bad_lines="skip")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df = df.sample(2000)

In [5]:
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df['label'] = df['label'].astype(int)

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['clean_text'] = df['review'].apply(clean_text)

In [7]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [9]:
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)

In [10]:
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = [int(x) for x in labels]

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [11]:
train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

In [12]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1
)

In [14]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    optimizers=(optimizer, None)
)

In [16]:
trainer.train()

C:\Users\jitendra khandelwal\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=200, training_loss=0.5065151596069336, metrics={'train_runtime': 2040.44, 'train_samples_per_second': 0.784, 'train_steps_per_second': 0.098, 'total_flos': 105244422144000.0, 'train_loss': 0.5065151596069336, 'epoch': 1.0})

In [17]:
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=1)

acc1 = accuracy_score(test_labels, y_pred)

print("Accuracy:", acc1)
print("Precision:", precision_score(test_labels, y_pred))
print("Recall:", recall_score(test_labels, y_pred))
print("F1 Score:", f1_score(test_labels, y_pred))
print("Confusion_matrix:",confusion_matrix(test_labels, y_pred))

C:\Users\jitendra khandelwal\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy: 0.825
Precision: 0.8315789473684211
Recall: 0.8061224489795918
F1 Score: 0.8186528497409327
Confusion_matrix: [[86 16]
 [19 79]]


In [18]:
for param in model.bert.parameters():
    param.requires_grad = False

trainer.train()

C:\Users\jitendra khandelwal\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=200, training_loss=0.3308926773071289, metrics={'train_runtime': 914.3986, 'train_samples_per_second': 1.75, 'train_steps_per_second': 0.219, 'total_flos': 105244422144000.0, 'train_loss': 0.3308926773071289, 'epoch': 1.0})

In [19]:
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=1)

acc2 = accuracy_score(test_labels, y_pred)
print("Frozen Accuracy:", acc2)

C:\Users\jitendra khandelwal\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Frozen Accuracy: 0.84


In [20]:
for name, param in model.bert.named_parameters():
    if "layer.10" in name or "layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainer.train()

C:\Users\jitendra khandelwal\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=200, training_loss=0.3244664764404297, metrics={'train_runtime': 1077.5133, 'train_samples_per_second': 1.485, 'train_steps_per_second': 0.186, 'total_flos': 105244422144000.0, 'train_loss': 0.3244664764404297, 'epoch': 1.0})

In [21]:
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=1)

acc3 = accuracy_score(test_labels, y_pred)
print("Last 2 Layers Accuracy:", acc3)

C:\Users\jitendra khandelwal\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Last 2 Layers Accuracy: 0.835


In [22]:
comparison = pd.DataFrame({
    "Experiment": ["Full Fine-tune", "Frozen BERT", "Last 2 Layers"],
    "Accuracy": [acc1, acc2, acc3]
})

comparison

,Experiment,Accuracy
0,Full Fine-tune,0.825
1,Frozen BERT,0.840
2,Last 2 Layers,0.835


### Final Analysis

- Full fine-tuning achieved highest accuracy.
- Frozen BERT reduced performance due to no weight updates.
- Fine-tuning last layers balanced accuracy and efficiency.
- BERT works well due to contextual embeddings.